<a href="https://colab.research.google.com/github/rawanmd/segmentation_coverless_steg/blob/adaptive_window/Segmentation_Coverless_Steg.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
from textwrap import indent
from google.colab import drive

import os
import glob
import json

In [10]:
INVERTED_INDEX_FILEPATH : str = "/content/drive/MyDrive/coco_dataset/inverted_index.json"

## Connect to Drive

In [2]:
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Dataset root
dataset_dir = "/content/drive/MyDrive/coco_dataset"
os.makedirs(dataset_dir, exist_ok=True)

# Move into dataset folder
%cd $dataset_dir

# Create desired folders
os.makedirs("annotations", exist_ok=True)
os.makedirs("train", exist_ok=True)
os.makedirs("val", exist_ok=True)

# ==========================
# 1. ANNOTATIONS
# ==========================
# !wget -c http://images.cocodataset.org/annotations/annotations_trainval2017.zip

# !unzip -q annotations_trainval2017.zip
# !rm annotations_trainval2017.zip

# # ==========================
# # 2. VALIDATION IMAGES
# # ==========================
# !wget -c http://images.cocodataset.org/zips/val2017.zip

# !unzip -q val2017.zip
# !rm val2017.zip

# # Move images into val/
# !mv val2017/* val/
# !rmdir val2017

# ==========================
# 3. TRAIN IMAGES
# ==========================
# !wget -c http://images.cocodataset.org/zips/train2017.zip

# !unzip -q train2017.zip
# !rm train2017.zip

# # Move images into train/
# !mv train2017/* train/
# !rmdir train2017

Mounted at /content/drive
/content/drive/MyDrive/coco_dataset


## Import SAM

In [ ]:
!pip install git+https://github.com/facebookresearch/segment-anything.git
!pip install opencv-python pycocotools matplotlib
!pip install torch torchvision

import torch
import cv2
import matplotlib.pyplot as plt

!wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-3t1woizo
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-3t1woizo
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36592 sha256=0ff9fd0b047c3b2576d90fa6dffa4d8e2c9ff73f9d45aa79c23ac37115871cbe
  Stored in directory: /tmp/pip-ephem-wheel-cache-kmqoe5ft/wheels/29/82/ff/04e2be9805a1cb48bec0b85b5a6da6b63f647645750a0e42d4
Successfully built segment_anything
--2026-08-11 14:30:41--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.9.168.62, 65.9.168.52, 65.9.168.81, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.9.168.62|

In [ ]:
MODEL_TYPE = "vit_b"
CHECKPOINT = "sam_vit_b_01ec64.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"

sam = sam_model_registry[MODEL_TYPE](checkpoint=CHECKPOINT)
sam.to(device=device)

mask_generator = SamAutomaticMaskGenerator(sam)

## Functions

In [11]:
def extract_masks(image_path):

  import math

  image = cv2.imread(image_path)
  image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
  masks = mask_generator.generate(image)

  binary_seq = ""

  for i in range(len(masks) - 1):
    bbox1 = masks[i]['bbox']
    x1, y1 = bbox1[:2]

    bbox2 = masks[i+1]['bbox']
    x2, y2 = bbox2[:2]

    d_squared = (x2 - x1)**2 + (y2 - y1)**2
    d = math.sqrt(d_squared)

    if(d <= 300):
      binary_seq += '0'
    else:
      binary_seq += '1'

  return binary_seq, len(binary_seq)

def get_image_paths(dir):
  image_paths = sorted(glob.glob(os.path.join(dir, '*.jpg')))
  print(f"Found {len(image_paths)} images.")
  return image_paths

def build_index(image_paths):
  index = {}

  for image_path in image_paths:
    # call segmentation model
    binary_seq, length = extract_masks(image_path)

    if binary_seq is None:
      continue

    if binary_seq not in index:
      index[binary_seq] = []

    index[binary_seq].append({"image path": image_path,
                              "length": length})

    with open(str(dataset_dir)+"/inverted_index.json", "w") as f:
      json.dump(index, f, indent=4)

  return index

In [16]:
def match_msg_to_images(binary_msg: str, window_size: int, inverted_index: dict) -> (list[str], int, int):
  """
  finds the optimal window size and matches images in the inverted index with the binary representation of the message.

  Searches using an index with variable-length keys.The scan window is sized to the longest key in the index, ensuring
  that all possible key matches can be detected.

  Args:
    binary_msg (str): Binary representation of the secret message.
    window_size (int): Maximum window size available in the inverted index.
    inverted_index (dict): contains all binary_sequences extracted from the images in the dataset and the images themselves.

  Returns:
    image_paths (list[str]): list of file paths of images that matched the binary message. Returns [] if matching failed.
    window_size (int): Final window size that successfully matched the entire message. Returns -1 if matching failed.
    num_zeros (int): the number of trailing zeroes that were added to the last chunk match the window size or -1 if matching failed.
  """
  for current_window_size in  range(window_size, 0, -1):

      image_paths = []
      success = True

      print(f"\nTrying window size: {window_size}")

      for i in range(0, len(binary_msg), window_size):

          chunk = binary_msg[i:i + window_size]
          num_zeros = 0

          # add trailing zeros to last chunk
          if len(chunk) < window_size:
              num_zeros = window_size - len(chunk)
              chunk = chunk.ljust(window_size, '0')

          print(f"chunk: {chunk}")

          key = ''

          for idx in inverted_index:
              if idx.startswith(chunk):
                  key = idx
                  break

          if key == '':
              print(f"No match for {chunk}")
              success = False
              break

          image_paths.append(inverted_index[key][0]['image path'])
          print(f"key: {key}")

      if success:
          print(f"image paths: {image_paths}")
          print(f"Final window size: {window_size}")
          return image_paths, window_size, num_zeros

      window_size -= 1

  return [], -1, -1

In [17]:
def string_to_binary(text: str) -> str:
    """
    Converts a string into a space-separated binary representation.
    """
    return ' '.join(format(ord(char), '08b') for char in text)

def binary_to_string(binary_text: str) -> str:
    """
    Converts a space-separated binary string back into text.
    """
    return ''.join(chr(int(binary_val, 2)) for binary_val in binary_text.split())

## Pipeline

In [18]:
PATH = "/content/drive/MyDrive/coco_dataset/train2017"

secret_msg = "Hello World"

binary_msg = string_to_binary(secret_msg)

# build index if not already there
# inverted_index = build_index(PATH)

# load index if it exists
with open(INVERTED_INDEX_FILEPATH, "r") as f:
  inverted_index = json.load(f)

In [19]:
window_size = len(max(inverted_index.keys(), key=len))
print(f"Max window size in index: {window_size}")

# all_keys, w_size, num_zeros = match_msg_to_images(binary_msg, window_size, inverted_index)
all_keys, w_size, num_zeros = match_msg_to_images('1101111', 4, inverted_index)  # just an example for tracing

Max window size in index: 309

Trying window size: 4
chunk: 1101
key: 11011011111100110110110110001101000
chunk: 1110
key: 1110001100011010001110110010010000111100100000011001100000111111011010
image paths: ['/content/drive/MyDrive/coco_dataset/train2017/000000000208.jpg', '/content/drive/MyDrive/coco_dataset/train2017/000000000321.jpg']
Final window size: 4
